# Setup

In [7]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import logging
# logging.basicConfig(format="%(asctime)s - %(name)s - %(levelname)s - %(message)s")
logging.basicConfig(format="%(levelname)s:%(name)s:  %(message)s")

import sentencepiece as spm

from vocab import Vocab
from utils import read_corpus, batch_iter
from nmt_model import NMT

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
logging.getLogger("nmt").setLevel(logging.DEBUG)

In [ ]:
logging.getLogger("nmt").setLevel(logging.INFO)

# 01 Data Preparation

## 01 Padding sequences

## 02 Vocabulary

In [21]:
# Load the vocabulary from the provided file
vocab = Vocab.load("vocab.json")

In [ ]:
# Print the vocabulary object representation to verify it has been loaded correctly
print(vocab)

Vocab(source 21001 words, target 8001 words)


In [7]:
# Print the sizes of the source and target vocabularies
print("Source vocabulary size:", len(vocab.src))
print("Target vocabulary size:", len(vocab.tgt))

Source vocabulary size: 21001
Target vocabulary size: 8001


In [8]:
# Print the IDs of special tokens in the source vocabulary
print("Source PAD ID:", vocab.src["<pad>"])
print("Source BOS ID:", vocab.src["<s>"])
print("Source EOS ID:", vocab.src["</s>"])
print("Source UNK ID:", vocab.src["<unk>"])

Source PAD ID: 0
Source BOS ID: 1
Source EOS ID: 2
Source UNK ID: 3


Let's explore some methods of `VocabEntry` class.

In [13]:
# Print the types of the word2id and id2word dictionaries in the source vocabulary
print(f"Type of word2id dictionary: {type(vocab.src.word2id)}")
print(f"Type of id2word dictionary: {type(vocab.src.id2word)}")

# Print their sizes to verify they are consistent with the vocabulary size
print(f"Size of word2id dictionary: {len(vocab.src.word2id)}")
print(f"Size of id2word dictionary: {len(vocab.src.id2word)}")

# Print the first 5 entries of the word2id and id2word dictionaries to verify their contents
print("First 5 entries of word2id dictionary:", list(vocab.src.word2id.items())[:5])
print("First 5 entries of id2word dictionary:", list(vocab.src.id2word.items())[:5])

Type of word2id dictionary: <class 'dict'>
Type of id2word dictionary: <class 'dict'>
Size of word2id dictionary: 21001
Size of id2word dictionary: 21001
First 5 entries of word2id dictionary: [('<pad>', 0), ('<s>', 1), ('</s>', 2), ('<unk>', 3), (',', 4)]
First 5 entries of id2word dictionary: [(0, '<pad>'), (1, '<s>'), (2, '</s>'), (3, '<unk>'), (4, ',')]


In [16]:
# Create a sample list of sentences to check the conversion methods
sample_sentences = [
    ["<s>", "i", "am", "a", "student", "</s>"],
    ["<s>", "hello", "world", "</s>"]
]

# Use the `words2indices` method to convert the sample sentences to indices
sample_indices = vocab.tgt.words2indices(sample_sentences)
print(f"Sample sentences converted to indices: {sample_indices}")

Sample sentences converted to indices: [[1, 177, 1035, 165, 3, 2], [1, 3, 3, 2]]


We see here a lot of unknown tokens. It turns out that vocabulary can contain tokens like "_student", not just "student". So we need to use tokenizer to split the words into subwords before converting them to indices.

In [ ]:
tgt_sp = spm.SentencePieceProcessor(model_file="tgt.model")

sample_texts = [
    "i am a student",
    "hello world",
]

sample_pieces = []

for text in sample_texts:
    pieces = tgt_sp.encode(text, out_type=str)
    pieces = ["<s>", *pieces, "</s>"]
    sample_pieces.append(pieces)

# Convert the sample pieces to indices using the vocabulary
sample_indices = vocab.tgt.words2indices(sample_pieces)

for text, pieces, indices in zip(
    sample_texts,
    sample_pieces,
    sample_indices,
):
    print(f"text:   {text}")
    print(f"pieces: {pieces}")
    print(f"ids:    {indices}")
    print()

text:   i am a student
pieces: ['<s>', '▁i', '▁am', '▁a', '▁student', '</s>']
ids:    [1, 36, 282, 12, 3158, 2]

text:   hello world
pieces: ['<s>', '▁hello', '▁world', '</s>']
ids:    [1, 7747, 108, 2]



In [ ]:
# sample_indices is a list of lists, where each inner list contains IDs
type(sample_indices), type(sample_indices[0]), sample_indices[0]

(list, list, [1, 36, 282, 12, 3158, 2])

Finally, let's convert these samples into tensors using `to_input_tensor`.

In [20]:
sample_indices

[[1, 36, 282, 12, 3158, 2], [1, 7747, 108, 2]]

In [21]:
# Finally, let's convert these samples into tensors using `to_input_tensor`.
vocab.tgt.to_input_tensor(sample_pieces, device="cpu")

tensor([[   1,    1],
        [  36, 7747],
        [ 282,  108],
        [  12,    2],
        [3158,    0],
        [   2,    0]])

## 03 Utils

Let's check the `read_corpus` function. It reads a file and returns a list of sentences, where each sentence is a list of tokens. The function also adds special tokens like `<s>` and `</s>` to the beginning and end of each sentence.

In [6]:
# Specify a filename for the corpus to be read
filename = Path("zh_en_data/train_1K.en")

# Use the `read_corpus` function to read the corpus from the specified file
corpus = read_corpus(file_path=filename, source="tgt")

In [11]:
# Corpus is list[list[str]], where each inner list is a sentence represented as a list of tokens.
type(corpus), len(corpus), type(corpus[0]), len(corpus[0]), type(corpus[0][0])

(list, 1000, list, 24, str)

In [12]:
# Extract the first sentence from the corpus for demonstration
first_sentence = corpus[0]

# Print the first sentence to verify its contents
print(f"First sentence in the corpus: {first_sentence}")

First sentence in the corpus: ['<s>', '▁he', '▁was', '▁seen', '▁wear', 'ing', '▁a', '▁white', '▁', 'jacket', ',', '▁a', '▁dark', '-', 'coloured', '▁t', '-', 'shi', 'rt', '▁and', '▁p', 'ants', '.', '</s>']


In [13]:
# Read the training data from the source and target files using the `read_corpus` function
train_data_src = read_corpus("zh_en_data/train_1K.zh", source="src")
train_data_tgt = read_corpus("zh_en_data/train_1K.en", source="tgt")

# Convert train data into list of tuples (src_sent, tgt_sent)
train_data = list(zip(train_data_src, train_data_tgt))

In [16]:
len(train_data)

1000

In [19]:
train_data[0][0][:5], train_data[0][1][:5]

(['▁', '事发', '时', '身穿', '白色'], ['<s>', '▁he', '▁was', '▁seen', '▁wear'])

# 02 Model Implementation

In [3]:
# Load some data for testing the `read_corpus` function with a smaller dataset
train_data_src = read_corpus("zh_en_data/train_1K.zh", source='src', vocab_size=21000) # List[list[str]]
train_data_tgt = read_corpus("zh_en_data/train_1K.en", source='tgt', vocab_size=8000) # List[list[str]]

# Convert train and dev data into list of tuples (src_sent, tgt_sent)
train_data = list(zip(train_data_src, train_data_tgt)) # List[Tuple[List[str], List[str]]]

# Load the vocabulary from the provided file
vocab = Vocab.load("vocab.json")

# Fetch a batch of data using the `batch_iter` function to verify its functionality
batch_size = 2
src_sents, tgt_sents = next(batch_iter(train_data, batch_size=batch_size, shuffle=False))

# Compute the lengths of the source sentences in the batch BEFORE padding
source_lengths = [len(sentence) for sentence in src_sents]

# Convert list of lists into tensors
source_padded = vocab.src.to_input_tensor(src_sents, device="cpu")  # Shape: [src_len, batch_size]
target_padded = vocab.tgt.to_input_tensor(tgt_sents, device="cpu")  # Shape: [tgt_len, batch_size]

# Print the shapes of the padded source and target tensors to verify they are as expected
print(f"Source padded shape: {source_padded.shape}")  # Expected: [src_len, batch_size]
print(f"Target padded shape: {target_padded.shape}")  #Expected: [tgt_len, batch_size]

# Create a model instance for testing the `batch_iter` function
model = NMT(embed_size=1024,
            hidden_size=768,
            dropout_rate=0.2,
            vocab=vocab)

Source padded shape: torch.Size([19, 2])
Target padded shape: torch.Size([34, 2])


In [16]:
768 * 2

1536

In [18]:
# Run model.encode on the padded source and target tensors to check shapes
model.encode(source_padded, source_lengths);

DEBUG:nmt.model:  ====Running encode()=====
DEBUG:nmt.model:  source_padded shape: torch.Size([19, 2]), source_lengths: [19, 17]
DEBUG:nmt.model:  (1) X shape after embedding: torch.Size([19, 2, 1024])
DEBUG:nmt.model:  (2) X shape after post_embed_cnn: torch.Size([19, 2, 1024])
DEBUG:nmt.model:  (3) enc_hiddens shape after encoder: torch.Size([19, 2, 1536])
DEBUG:nmt.model:  (3) last_hidden shape: torch.Size([2, 2, 768]), last_cell shape: torch.Size([2, 2, 768])
DEBUG:nmt.model:  (3) enc_hiddens shape after permute: torch.Size([2, 19, 1536])
DEBUG:nmt.model:  (4) last_hidden shape after concatenation: torch.Size([2, 1536])
DEBUG:nmt.model:  (4) last_cell shape after concatenation: torch.Size([2, 1536])
